# Week 102, Day 5: Practice final

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w102d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


*Interactive figure on the website: Two hours. The timer survives reloads; stopping is on your honour..*

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: P1: Reading a route description

Each line of a route description reads `FROM to TO: KM km` (for example `Mill to Weir: 1.5 km`); blank lines and lines starting with `#` are ignored. Write `parse_legs(text)`: a list of `(from, to, km)` with km a float; raise `ValueError("line N: can't read LINE")` (N counting every line from 1) for any other line.

In [ ]:
import re

def parse_legs(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_legs():\n    \"strings and errors\"\n    text = \"# Ridge walk\\nMill to Weir: 1.5 km\\n\\nWeir to Farm: 2.5 km\\nFarm to Ridge: 2 km\\n\"\n    assert parse_legs(text) == [(\"Mill\", \"Weir\", 1.5), (\"Weir\", \"Farm\", 2.5), (\"Farm\", \"Ridge\", 2.0)]\n    try:\n        parse_legs(\"Mill to Weir: 1.5 km\\nWeir - Farm 2.5\")\n    except ValueError as e:\n        assert str(e) == \"line 2: can't read Weir - Farm 2.5\"\n    else:\n        raise AssertionError(\"bad line\")\n    assert parse_legs(\"\") == []")

### Exercise: P2: The busiest hour

Bookings arrive at `minutes` past midnight (unsorted). Write `busiest_hour(minutes)`: `(start, count)` for the 60-minute window `[start, start + 59]` containing the most bookings, where `start` is a booking's own time (the earliest on ties); `(None, 0)` if there are none. It must run in O(n log n).

In [ ]:
def busiest_hour(minutes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\n\ndef test_busiest():\n    \"sort, then a window\"\n    assert busiest_hour([600, 610, 659, 660, 700, 30]) == (600, 3) and busiest_hour([]) == (None, 0) and busiest_hour([5]) == (5, 1)\n    assert busiest_hour([0, 59, 60, 119]) == (0, 2)\n\ndef test_fast():\n    \"100,000 bookings\"\n    rng = random.Random(3)\n    times = [rng.randint(0, 1439) for _ in range(100_000)]\n    start = time.perf_counter()\n    s, c = busiest_hour(times)\n    assert time.perf_counter() - start < 2 and c == sum(1 for t in times if s <= t <= s + 59)")

### Exercise: P3: A stock of kit

Write a class `Kit` for a club's kit store: `add(item, n=1)`, `take(item, n=1)` (raising `KeyError("no ITEM")` for an unknown item and `ValueError("only N ITEM left")` if not enough; items reaching 0 are removed), `kit["map"]` (the count, 0 for unknown), `len(kit)` (distinct items), iteration in alphabetical order of item, and `repr(kit)` like `Kit(flask=2, map=5)`.

In [ ]:
class Kit:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_kit():\n    \"objects with the data model\"\n    k = Kit()\n    k.add(\"map\", 5)\n    k.add(\"flask\", 2)\n    k.add(\"map\")\n    k.take(\"map\", 2)\n    assert k[\"map\"] == 4 and k[\"stove\"] == 0 and len(k) == 2 and list(k) == [\"flask\", \"map\"] and repr(k) == \"Kit(flask=2, map=4)\"\n    k.take(\"flask\", 2)\n    assert list(k) == [\"map\"]\n    for action, kind, message in [(lambda: k.take(\"stove\"), KeyError, \"'no stove'\"), (lambda: k.take(\"map\", 9), ValueError, \"only 4 map left\")]:\n        try:\n            action()\n        except kind as e:\n            assert str(e) == message\n        else:\n            raise AssertionError(message)")

### Exercise: P4: A lazy log reader

Write a generator `records(lines)` that yields a dict `{"day": ..., "km": float, "minutes": int}` for each line `day,km,minutes`, skipping blank lines and lines starting with `#`; for a line it can't convert, add the note `"line N"` (N counting from 1) to the `ValueError` and re-raise it. Then `fastest(lines)`: the record with the lowest minutes per km (first on ties), reading the lines once.

In [ ]:
def records(lines):
    ___

def fastest(lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import inspect\n\ndef test_records():\n    \"lazy, with context in errors\"\n    lines = [\"# walks\", \"2026-01-04,6.0,105\", \"\", \"2026-01-18,7.5,98\", \"2026-02-01,11.0,150\"]\n    assert inspect.isgenerator(records(lines)) and len(list(records(lines))) == 3\n    assert fastest(lines) == {\"day\": \"2026-01-18\", \"km\": 7.5, \"minutes\": 98}\n    try:\n        list(records([\"2026-01-04,6.0,105\", \"2026-01-05,six,90\"]))\n    except ValueError as e:\n        assert e.__notes__ == [\"line 2\"]\n    else:\n        raise AssertionError(\"bad line\")")

### Exercise: P5: Signed share links

Write `share_link(route_id, expires, key)` returning `"ROUTE_ID.EXPIRES.SIGNATURE"` (the signature the hex HMAC-SHA256 of `"ROUTE_ID.EXPIRES"` with `key`), and `open_link(link, now, key)` returning `(status, data)`: `(200, {"route": ROUTE_ID})` for a valid, unexpired link (`now <= expires`, as ints), `(410, {"error": "expired"})` for a valid but expired one, and `(403, {"error": "invalid link"})` for anything malformed or with a wrong signature (compare with `hmac.compare_digest`).

In [ ]:
import hashlib
import hmac

def share_link(route_id, expires, key):
    ___

def open_link(link, now, key):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_links():\n    \"integrity and expiry\"\n    key = b\"exam-\" + b\"practice-key\"\n    link = share_link(17, 1_800_000_000, key)\n    assert open_link(link, 1_700_000_000, key) == (200, {\"route\": 17})\n    assert open_link(link, 1_900_000_000, key) == (410, {\"error\": \"expired\"})\n    rid, exp, sig = link.split(\".\")\n    assert open_link(f\"18.{exp}.{sig}\", 1_700_000_000, key) == (403, {\"error\": \"invalid link\"})\n    assert open_link(f\"{rid}.1999999999.{sig}\", 1_700_000_000, key) == (403, {\"error\": \"invalid link\"})\n    assert open_link(\"nonsense\", 0, key) == (403, {\"error\": \"invalid link\"}) and open_link(link, 0, b\"other\") == (403, {\"error\": \"invalid link\"})")

### Exercise: C: Booking guided walks

Write `Bookings` for guided walks. `add_walk(walk_id, places)` creates a walk (an existing id raises `ValueError("walk WALK_ID exists")`). `book(walk_id, name)` returns `"booked"` if a place is free, `"waiting"` (joining the waiting list, first come first served) if not, `"already booked"` if the name is booked or waiting; an unknown walk raises `KeyError`. `cancel(walk_id, name)` removes the name from the booked list or the waiting list and, if a booked place was freed, books the first person waiting, returning their name (or `None`). `status(walk_id)` returns `{"booked": [...], "waiting": [...], "free": N}`. `to_json()` and `Bookings.from_json(text)` round-trip everything.

In [ ]:
import json

class Bookings:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bookings():\n    \"classes, queues, formats\"\n    b = Bookings()\n    b.add_walk(\"ridge\", 2)\n    assert [b.book(\"ridge\", n) for n in (\"ada\", \"bo\", \"cy\", \"di\", \"ada\")] == [\"booked\", \"booked\", \"waiting\", \"waiting\", \"already booked\"]\n    assert b.status(\"ridge\") == {\"booked\": [\"ada\", \"bo\"], \"waiting\": [\"cy\", \"di\"], \"free\": 0}\n    assert b.cancel(\"ridge\", \"ada\") == \"cy\" and b.cancel(\"ridge\", \"di\") is None\n    assert b.status(\"ridge\") == {\"booked\": [\"bo\", \"cy\"], \"waiting\": [], \"free\": 0}\n    b.cancel(\"ridge\", \"bo\")\n    assert b.status(\"ridge\")[\"free\"] == 1\n    copy = Bookings.from_json(b.to_json())\n    assert copy.status(\"ridge\") == b.status(\"ridge\") and copy.book(\"ridge\", \"ed\") == \"booked\"\n    for action, kind in [(lambda: b.add_walk(\"ridge\", 5), ValueError), (lambda: b.book(\"lake\", \"x\"), KeyError)]:\n        try:\n            action()\n        except kind:\n            pass\n        else:\n            raise AssertionError(kind)")

## How to use this

Treat the next two hours as the real thing: notes closed, no web, timer on. Run your code and press **Check** as often as you like. Each problem has a hint and a model solution: **don't open them until the timer has finished**.

**The marks.**

| Part | What | Marks |
|---|---|---|
| A | 10 quiz questions | 10 (1 each) |
| B | 5 problems | 70 |
| C | one larger program | 30 |
| | **Total** | **110** |

Count full marks for a problem only when every check passes. 66 (60%) is a pass, 88 (80%) a distinction. The real final is the same idea, larger: Part 1 (a quiz and problems) on the first day, Part 2 (a larger program and a review task) on the second.

## Part A: quiz

## Part B: five problems

## Part C: a larger program

## Afterwards

When the timer ends, add up your marks, then go back through each problem with its hint and solution. For every mark you dropped, write one line: the problem, the reason (**didn't know it**, **misread it**, **an edge case**, **ran out of time**) and the week to revise.

Which ideas did today need? P1 regular expressions and errors (Weeks 11 and 23); P2 sorting and a sliding window (Weeks 18 and 41); P3 the data model (Weeks 28 to 31); P4 generators and exception notes (Weeks 59 and 37); P5 HMAC and status codes (Weeks 76 to 79); C classes, queues and JSON together (Phases 1 to 3 and 4).

If the score disappointed you, spend tomorrow redoing the problems you dropped marks on, without looking, until they pass. Then rest: the day before the exam is for rest.